In [ ]:
# 1. Nuke the clashing Colab default PyTorch
!pip uninstall -y torch torchvision torchaudio fastai

# 2. Install the CPU-only version of PyTorch to prevent NCCL crashes
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# 3. Install PaddlePaddle GPU
!pip install paddlepaddle-gpu

# 4. Install the EXACT older version of PaddleOCR that still has PPStructure
!pip install paddleocr==2.8.1 pymupdf "paddleocr[all]==2.8.1"

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
Found existing installation: fastai 2.8.7
Uninstalling fastai-2.8.7:
  Successfully uninstalled fastai-2.8.7
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 543.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Fo

In [ ]:
# Install the missing HTML formatting library
!pip install premailer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 13.7 MB/s eta 0:00:00


In [ ]:
import os

# Define your paths
INPUT_DIR = '/content/my_images'
OUTPUT_DIR = '/content/my_annotations'

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"1. Upload your images to: {INPUT_DIR}")
print(f"2. Your JSONs will appear in: {OUTPUT_DIR}")

1. Upload your images to: /content/my_images
2. Your JSONs will appear in: /content/my_annotations


In [ ]:
import os
import json
import cv2
import numpy as np
import re
from paddleocr import PaddleOCR

# Initialize OCR (use_gpu=True works now because we cleared the PyTorch clash)
ocr = PaddleOCR(lang='en', use_gpu=True, show_log=False)

def get_word_boxes_from_line(line_img, words):
    line_w = line_img.shape[1]
    if len(words) <= 1 or line_w <= 0:
        return [0.0], [float(line_w)]

    if len(line_img.shape) == 3:
        gray = cv2.cvtColor(line_img, cv2.COLOR_BGR2GRAY)
    else:
        gray = line_img

    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    projection = np.sum(binary, axis=0)
    gaps = np.where(projection == 0)[0]

    if len(gaps) == 0:
        total_chars = sum(len(w) for w in words)
        starts, ends = [], []
        curr = 0.0
        for w in words:
            w_width = (len(w) / total_chars) * line_w
            starts.append(curr)
            ends.append(curr + w_width)
            curr += w_width
        return starts, ends

    gap_groups = np.split(gaps, np.where(np.diff(gaps) != 1)[0] + 1)
    gap_groups = sorted(gap_groups, key=len, reverse=True)

    if len(gap_groups) < len(words) - 1:
        avg_w = line_w / len(words)
        return [i * avg_w for i in range(len(words))], [(i + 1) * avg_w for i in range(len(words))]

    split_gaps = sorted(gap_groups[:len(words)-1], key=lambda x: x[0])
    split_xs = [np.mean(g) for g in split_gaps]
    return [0.0] + split_xs, split_xs + [float(line_w)]

def get_tight_word_boxes(img_path):
    image = cv2.imread(img_path)
    if image is None: return None
    h, w, _ = image.shape
    result = ocr.ocr(img_path, cls=True)
    all_tokens, all_bboxes = [], []

    if result and result[0]:
        for line in result[0]:
            box, (text, score) = line[0], line[1]
            words = [word for word in re.split(r'(:|\s+)', text) if word.strip()]
            line_xmin = max(0, int(min([p[0] for p in box])))
            line_ymin = max(0, int(min([p[1] for p in box])))
            line_xmax = min(w, int(max([p[0] for p in box])))
            line_ymax = min(h, int(max([p[1] for p in box])))

            if len(words) == 1:
                all_tokens.append(words[0])
                all_bboxes.append([int(1000*line_xmin/w), int(1000*line_ymin/h), int(1000*line_xmax/w), int(1000*line_ymax/h)])
            else:
                line_crop = image[line_ymin:line_ymax, line_xmin:line_xmax]
                starts, ends = get_word_boxes_from_line(line_crop, words)
                for i, word in enumerate(words):
                    all_tokens.append(word)
                    all_bboxes.append([
                        int(1000*(line_xmin+starts[i])/w), int(1000*line_ymin/h),
                        int(1000*(line_xmin+ends[i])/w), int(1000*line_ymax/h)
                    ])
    return {"file_name": os.path.basename(img_path), "tokens": all_tokens, "bboxes": all_bboxes, "ner_tags": [0]*len(all_tokens)}

# --- EXECUTION LOOP ---
valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_exts)]

print(f"Found {len(files)} images. Starting processing...")

for filename in files:
    img_path = os.path.join(INPUT_DIR, filename)
    output_filename = os.path.splitext(filename)[0] + ".json"
    output_path = os.path.join(OUTPUT_DIR, output_filename)

    try:
        data = get_tight_word_boxes(img_path)
        if data:
            with open(output_path, 'w') as f:
                json.dump(data, f, indent=4)
            print(f"✅ Processed: {filename}")
    except Exception as e:
        print(f"❌ Failed: {filename} | Error: {e}")

print("Done! You can download your files from the output folder.")

download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 4.00M/4.00M [00:00<00:00, 6.01MiB/s]


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10.2M/10.2M [00:00<00:00, 11.6MiB/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:00<00:00, 4.11MiB/s]


Found 42 images. Starting processing...
[2026/03/26 04:48:31] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
✅ Processed: invoice_ISI007532198_page0.png
[2026/03/26 04:48:33] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
✅ Processed: invoice_CPI007527460_page0.png
[2026/03/26 04:48:34] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
✅ Processed: invoice_CPI007532671_page0.png
[2026/03/26 04:48:34] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
✅ Processed: invoice_CPI007529709_page0.png
[2026/03/26 04:48:35] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
✅ Processed: invoice_CPI007532659_page0.png
[2026/03/26 04:48:36] ppocr WARNING: Since the angle classifier is not initialized, i

In [ ]:
!zip -r annotations.zip /content/my_annotations
from google.colab import files
files.download('annotations.zip')

  adding: content/my_annotations/ (stored 0%)
  adding: content/my_annotations/invoice_CPI007529714_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007529621_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007527454_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007530918_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007529595_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007527460_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007529614_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007534830_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007527461_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007527474_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPI007527264_page0.json (deflated 89%)
  adding: content/my_annotations/invoice_CPC000692500_page0.json (deflated

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import json
import cv2
import numpy as np
import re
from paddleocr import PaddleOCR

# 1. Initialize OCR
ocr = PaddleOCR(lang='en', use_gpu=True, show_log=False)

def get_word_boxes_from_line(line_img, words):
    """
    Splits a single line image into multiple word-level bounding boxes.
    """
    line_w = line_img.shape[1]

    # Safety check: If only one word or image is empty
    if len(words) <= 1 or line_w <= 0:
        return [0.0], [float(line_w)]

    # Convert to grayscale for processing
    if len(line_img.shape) == 3:
        gray = cv2.cvtColor(line_img, cv2.COLOR_BGR2GRAY)
    else:
        gray = line_img

    # Binarize to find text pixels (black text on white background)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    projection = np.sum(binary, axis=0)
    gaps = np.where(projection == 0)[0]

    # Fallback A: If NO gaps are found, split by character length ratio
    if len(gaps) == 0:
        total_chars = sum(len(w) for w in words)
        starts, ends = [], []
        curr = 0.0
        for w in words:
            w_width = (len(w) / total_chars) * line_w
            starts.append(curr)
            ends.append(curr + w_width)
            curr += w_width
        return starts, ends

    # Group consecutive gap pixels into chunks
    gap_groups = np.split(gaps, np.where(np.diff(gaps) != 1)[0] + 1)
    gap_groups = sorted(gap_groups, key=len, reverse=True)

    # Fallback B: If we didn't find enough physical gaps
    if len(gap_groups) < len(words) - 1:
        avg_w = line_w / len(words)
        return [i * avg_w for i in range(len(words))], [(i + 1) * avg_w for i in range(len(words))]

    # Success: Pick the widest gaps and sort them horizontally
    split_gaps = sorted(gap_groups[:len(words)-1], key=lambda x: x[0])
    split_xs = [np.mean(g) for g in split_gaps]

    starts = [0.0] + split_xs
    ends = split_xs + [float(line_w)]
    return starts, ends

def get_tight_word_boxes(img_path):
    image = cv2.imread(img_path)
    if image is None: return None
    h, w, _ = image.shape

    result = ocr.ocr(img_path, cls=True)
    all_tokens, all_bboxes = [], []

    if result and result[0]:
        for line in result[0]:
            box, (text, score) = line[0], line[1]

            # --- MODIFIED: Split by colon (:) or spaces ---
            # Using ( : | \s+ ) keeps the colon as its own token.
            # If you want to delete the colon, use [:\s]+
            words = [word for word in re.split(r'(:|\s+)', text) if word.strip()]

            line_xmin = max(0, int(min([p[0] for p in box])))
            line_ymin = max(0, int(min([p[1] for p in box])))
            line_xmax = min(w, int(max([p[0] for p in box])))
            line_ymax = min(h, int(max([p[1] for p in box])))

            if len(words) == 1:
                all_tokens.append(words[0])
                all_bboxes.append([int(1000*line_xmin/w), int(1000*line_ymin/h), int(1000*line_xmax/w), int(1000*line_ymax/h)])
            else:
                line_crop = image[line_ymin:line_ymax, line_xmin:line_xmax]
                starts, ends = get_word_boxes_from_line(line_crop, words)

                for i, word in enumerate(words):
                    all_tokens.append(word)
                    # Mapping local crop coordinates back to global normalized coordinates
                    all_bboxes.append([
                        int(1000*(line_xmin+starts[i])/w),
                        int(1000*line_ymin/h),
                        int(1000*(line_xmin+ends[i])/w),
                        int(1000*line_ymax/h)
                    ])

    return {
        "file_name": os.path.basename(img_path),
        "tokens": all_tokens,
        "bboxes": all_bboxes,
        "ner_tags": [0]*len(all_tokens)
    }

# --- EXECUTION ---
img_filename = 'invoice_CPC000692108_page0.png'
if os.path.exists(img_filename):
    data = get_tight_word_boxes(img_filename)
    with open('annotation_fixed.json', 'w') as f:
        json.dump(data, f, indent=4)
    print(f"Successfully processed {len(data['tokens'])} tokens!")
else:
    print(f"Error: {img_filename} not found in current directory.")


[2026/03/25 10:27:21] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
Successfully processed 294 tokens!
